In [0]:
!pip install kaggle

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os

os.environ["KAGGLE_USERNAME"] = "dnandi"
os.environ["KAGGLE_KEY"] = "KGAT_24e60868ea7a8fc8dc48dba5439a3692"

print("Kaggle credentials configured!")

Kaggle credentials configured!


In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.ecommerce
""")

DataFrame[]

In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS workspace.ecommerce.ecommerce_data
""")

DataFrame[]

In [0]:
%sh
cd /Volumes/workspace/ecommerce/ecommerce_data
kaggle datasets download -d mkechinov/ecommerce-behavior-data-from-multi-category-store

Dataset URL: https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store
License(s): copyright-authors


100%|██████████| 4.29G/4.29G [00:42<00:00, 107MB/s]


In [0]:
%sh
cd /Volumes/workspace/ecommerce/ecommerce_data
unzip -o ecommerce-behavior-data-from-multi-category-store.zip
ls -lh

Archive:  ecommerce-behavior-data-from-multi-category-store.zip
  inflating: 2019-Nov.csv            
  inflating: 2019-Oct.csv            
total 18G
-rwxrwxrwx 1 spark-d7432064-83b4-4e15-9a46-8c nogroup 8.4G Feb 22 16:35 2019-Nov.csv
-rwxrwxrwx 1 spark-d7432064-83b4-4e15-9a46-8c nogroup 5.3G Feb 22 16:37 2019-Oct.csv
drwxrwxrwx 2 nobody                           nogroup 4.0K Feb 22 16:29 delta
-rwxrwxrwx 1 spark-d7432064-83b4-4e15-9a46-8c nogroup 4.3G Feb 22 16:34 ecommerce-behavior-data-from-multi-category-store.zip
drwxrwxrwx 2 nobody                           nogroup 4.0K Feb 22 16:29 oct_delta
drwxrwxrwx 2 nobody                           nogroup 4.0K Feb 22 16:29 outputs


In [0]:
%sh
cd /Volumes/workspace/ecommerce/ecommerce_data
rm -f ecommerce-behavior-data-from-multi-category-store.zip
ls -lh

total 14G
-rwxrwxrwx 1 spark-d7432064-83b4-4e15-9a46-8c nogroup 8.4G Feb 22 16:35 2019-Nov.csv
-rwxrwxrwx 1 spark-d7432064-83b4-4e15-9a46-8c nogroup 5.3G Feb 22 16:37 2019-Oct.csv
drwxrwxrwx 2 nobody                           nogroup 4.0K Feb 22 16:29 delta
drwxrwxrwx 2 nobody                           nogroup 4.0K Feb 22 16:29 oct_delta
drwxrwxrwx 2 nobody                           nogroup 4.0K Feb 22 16:29 outputs


In [0]:
%restart_python

In [0]:
df_n = spark.read.csv("/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv")

In [0]:
df = spark.read.csv("/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv")

In [0]:
print(f"October 2019 - Total Events: {df.count():,}")
print("\n" + "="*60)
print("SCHEMA:")
print("="*60)
df.printSchema()

October 2019 - Total Events: 42,448,765

SCHEMA:
root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)
 |-- _c6: string (nullable = true)
 |-- _c7: string (nullable = true)
 |-- _c8: string (nullable = true)



In [0]:
print("\n" + "="*60)
print("SAMPLE DATA (First 5 rows):")
print("="*60)
df.show(5, truncate=False)


SAMPLE DATA (First 5 rows):
+-----------------------+----------+----------+-------------------+-----------------------------------+--------+------+---------+------------------------------------+
|_c0                    |_c1       |_c2       |_c3                |_c4                                |_c5     |_c6   |_c7      |_c8                                 |
+-----------------------+----------+----------+-------------------+-----------------------------------+--------+------+---------+------------------------------------+
|event_time             |event_type|product_id|category_id        |category_code                      |brand   |price |user_id  |user_session                        |
|2019-10-01 00:00:00 UTC|view      |44600062  |2103807459595387724|NULL                               |shiseido|35.79 |541312140|72d76fde-8bb3-4e00-8c23-a032dfed738c|
|2019-10-01 00:00:00 UTC|view      |3900821   |2053013552326770905|appliances.environment.water_heater|aqua    |33.20 |554748717|9333dfb

In [0]:
print("Current columns:", df.columns)
column_map = {
    "_c0": "event_time", "_c1": "event_type", "_c2": "product_id", 
    "_c3": "category_id", "_c4": "category_code", "_c5": "brand", 
    "_c6": "price", "_c7": "user_id", "_c8": "user_session"
}
for old_name, new_name in column_map.items():
    if old_name in df.columns:
        df = df.withColumnRenamed(old_name, new_name)

print("✅ Columns renamed!")
df.printSchema()
df.show(2)

Current columns: ['_c0', '_c1', '_c2', '_c3', '_c4', '_c5', '_c6', '_c7', '_c8']
✅ Columns renamed!
root
 |-- event_time: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- user_session: string (nullable = true)

+--------------------+----------+----------+-------------------+-------------+--------+-----+---------+--------------------+
|          event_time|event_type|product_id|        category_id|category_code|   brand|price|  user_id|        user_session|
+--------------------+----------+----------+-------------------+-------------+--------+-----+---------+--------------------+
|          event_time|event_type|product_id|        category_id|category_code|   brand|price|  user_id|        user_session|
|2019-10-01 00:00:...| 

In [0]:
delta_path = "/Volumes/workspace/ecommerce/ecommerce_data/oct_delta"
df.write.format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .save(delta_path)
print(f"Row count: {spark.read.format('delta').load(delta_path).count()}")
df.printSchema()

Row count: 42448765
root
 |-- event_time: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- user_session: string (nullable = true)



In [0]:
delta_path = "/Volumes/workspace/ecommerce/ecommerce_data/oct_delta"
print("Delta table ready at:", delta_path)
print(f"Initial row count: {spark.read.format('delta').load(delta_path).count()}")
spark.sql(f"DESCRIBE DETAIL delta.`{delta_path}`").select("numFiles", "sizeInBytes", "properties").show(truncate=False)

Delta table ready at: /Volumes/workspace/ecommerce/ecommerce_data/oct_delta
Initial row count: 42448765
+--------+-----------+-------------------------------------+
|numFiles|sizeInBytes|properties                           |
+--------+-----------+-------------------------------------+
|43      |1493399323 |{delta.enableDeletionVectors -> true}|
+--------+-----------+-------------------------------------+



In [0]:
dfn = spark.read.option("inferSchema", "true").csv("/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv", header=False)
dfn = (dfn.withColumnRenamed("_c0", "event_time")
         .withColumnRenamed("_c1", "event_type")
         .withColumnRenamed("_c2", "product_id")
         .withColumnRenamed("_c3", "category_id")
         .withColumnRenamed("_c4", "category_code")
         .withColumnRenamed("_c5", "brand")
         .withColumnRenamed("_c6", "price")
         .withColumnRenamed("_c7", "user_id")
         .withColumnRenamed("_c8", "user_session"))
delta_path = "/Volumes/workspace/ecommerce/ecommerce_data/oct_delta"
print("📈 Append 1: Full November data")
dfn.write.format("delta").mode("append").save(delta_path)
print("📈 Append 2: 1M row subset")
dfn.limit(1000000).write.format("delta").mode("append").save(delta_path)
print("📈 Append 3: 500K row subset")
dfn.limit(500000).write.format("delta").mode("append").save(delta_path)
print(f"✅ Total rows after 3 appends: {spark.read.format('delta').load(delta_path).count()}")

📈 Append 1: Full November data
📈 Append 2: 1M row subset
📈 Append 3: 500K row subset
✅ Total rows after 3 appends: 111450745


In [0]:
delta_path = "/Volumes/workspace/ecommerce/ecommerce_data/oct_delta"
print("📊 BEFORE OPTIMIZE - File fragmentation:")
spark.sql(f"DESCRIBE DETAIL delta.`{delta_path}`").select("numFiles", "sizeInBytes").show(truncate=False)

📊 BEFORE OPTIMIZE - File fragmentation:
+--------+-----------+
|numFiles|sizeInBytes|
+--------+-----------+
|113     |4133657837 |
+--------+-----------+



In [0]:
print("🔧 Running OPTIMIZE (file compaction only)...")
spark.sql(f"OPTIMIZE delta.`{delta_path}`")

print("\n📊 BEFORE OPTIMIZE:")
spark.sql(f"DESCRIBE DETAIL delta.`{delta_path}`").select("numFiles", "sizeInBytes").show(truncate=False)

print("\n📊 AFTER OPTIMIZE:")
spark.sql(f"DESCRIBE DETAIL delta.`{delta_path}`").select("numFiles", "sizeInBytes").show(truncate=False)

print("\n✅ OPTIMIZE complete!")
print("• Small files from appends compacted into optimal sizes ✓")
print("• Expect fewer files, larger average size after compaction")
print("• Query performance improves due to fewer file reads")

🔧 Running OPTIMIZE (file compaction only)...

📊 BEFORE OPTIMIZE:
+--------+-----------+
|numFiles|sizeInBytes|
+--------+-----------+
|17      |4496237517 |
+--------+-----------+


📊 AFTER OPTIMIZE:
+--------+-----------+
|numFiles|sizeInBytes|
+--------+-----------+
|17      |4496237517 |
+--------+-----------+


✅ OPTIMIZE complete!
• Small files from appends compacted into optimal sizes ✓
• Expect fewer files, larger average size after compaction
• Query performance improves due to fewer file reads


In [0]:
print("⏱️  Query performance test (user filter):")
%time
result = spark.read.format("delta").load(delta_path)\
  .filter("user_id = '54131214072d76fde-8bb3-4e00-8c23-a032dfed738c'")\
  .count()
print(f"✅ User events found: {result}")

⏱️  Query performance test (user filter):
CPU times: user 3 μs, sys: 0 ns, total: 3 μs
Wall time: 5.48 μs
✅ User events found: 0


In [0]:
from pyspark.sql import functions as F

delta_path = "/Volumes/workspace/ecommerce/ecommerce_data/oct_delta"
events = spark.read.format("delta").load(delta_path)
safe_price = F.when(
    (F.col("price").isNotNull()) & 
    (F.col("price") != "") & 
    (F.length(F.trim(F.col("price"))) > 0) &
    (F.col("price").rlike("^[0-9]+\\.?[0-9]*$")),
    F.regexp_replace(F.col("price"), "[^0-9.]", "").cast("double")
).otherwise(None)

features_df = events.groupBy("user_id").agg(
    F.count("*").alias("total_events"),
    F.count(F.when(F.col("event_type") == "purchase", 1)).alias("purchases"),
    F.sum(safe_price).alias("total_spent"),
    F.avg(safe_price).alias("avg_price"),
    F.countDistinct("product_id").alias("unique_products"),
    F.countDistinct("user_session").alias("sessions"),
    F.min("event_time").alias("first_event_date_raw"),
    F.max("event_time").alias("last_event_date_raw")
).orderBy(F.desc("total_events"))
print("✅ Features created")
features_df.show(10, truncate=False)
features_df.printSchema()

✅ Features created
+---------+------------+---------+------------------+------------------+---------------+--------+-----------------------+-----------------------+
|user_id  |total_events|purchases|total_spent       |avg_price         |unique_products|sessions|first_event_date_raw   |last_event_date_raw    |
+---------+------------+---------+------------------+------------------+---------------+--------+-----------------------+-----------------------+
|568778435|22929       |0        |5187451.93        |226.23978062715338|1937           |22542   |2019-11-08 03:11:13 UTC|2019-11-30 19:33:38 UTC|
|569335945|14810       |0        |1655685.840000004 |111.79512761647563|111            |14810   |2019-11-09 13:02:38 UTC|2019-11-30 20:11:45 UTC|
|512475445|13796       |0        |1557917.240000001 |112.92528559002616|175            |13474   |2019-10-01 08:43:20 UTC|2019-11-30 14:57:21 UTC|
|512365995|10295       |0        |3864168.7100000023|375.3442166100051 |352            |967     |2019-10-

In [0]:
silver_path = "/Volumes/workspace/ecommerce/ecommerce_data/silver/user_features"
final_features = features_df.dropDuplicates(["user_id"])

final_features.write.format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .save(silver_path)

print(f"✅ Silver layer saved: {silver_path}")
print(f"📊 Unique users: {spark.read.format('delta').load(silver_path).count()}")
spark.sql(f"OPTIMIZE delta.`{silver_path}` ZORDER BY (user_id)")
print("✅ Optimized!")

✅ Silver layer saved: /Volumes/workspace/ecommerce/ecommerce_data/silver/user_features
📊 Unique users: 5316650
✅ Optimized!


In [0]:
user_features = spark.read.format("delta").load(silver_path)

print("🏆 SILVER LAYER QUALITY REPORT")
print("\n📈 Dataset Stats:")
user_features.select(
    F.count("*").alias("total_users"),
    F.sum("total_events").alias("total_events"),
    F.sum("purchases").alias("total_purchases"),
    F.round(F.avg("total_spent"), 2).alias("avg_spent")
).show(truncate=False)

print("\n✅ Quality Checks:")
user_features.select(
    F.count(F.when(F.col("total_spent").isNull(), 1)).alias("null_spent"),
    F.count(F.when(F.col("purchases") > 0, 1)).alias("buyers"),
    F.approx_count_distinct("user_id").alias("unique_users")
).show(truncate=False)

print("\n🔥 Top 5 Power Users:")
user_features.select("user_id", "total_events", "purchases", "total_spent", "avg_price") \
  .orderBy(F.desc("total_events")).limit(5).show(truncate=False)

🏆 SILVER LAYER QUALITY REPORT

📈 Dataset Stats:
+-----------+------------+---------------+---------+
|total_users|total_events|total_purchases|avg_spent|
+-----------+------------+---------------+---------+
|5316650    |111450745   |1687200        |6113.32  |
+-----------+------------+---------------+---------+


✅ Quality Checks:
+----------+------+------------+
|null_spent|buyers|unique_users|
+----------+------+------------+
|1         |697470|5291243     |
+----------+------+------------+


🔥 Top 5 Power Users:
+---------+------------+---------+------------------+------------------+
|user_id  |total_events|purchases|total_spent       |avg_price         |
+---------+------------+---------+------------------+------------------+
|568778435|22929       |0        |5187451.93        |226.23978062715338|
|569335945|14810       |0        |1655685.840000004 |111.79512761647563|
|512475445|13796       |0        |1557917.240000001 |112.92528559002616|
|512365995|10295       |0        |3864168

In [0]:
silver_path = "/Volumes/workspace/ecommerce/ecommerce_data/silver/user_features"

print("🔍 SQL Queries on Silver Layer (Direct Path):")
print("\n1. Top buyers:")
spark.sql(f"""
  SELECT user_id, total_events, purchases, total_spent,
    ROUND(total_spent / NULLIF(purchases, 0), 2) as avg_order_value
  FROM delta.`{silver_path}`
  WHERE purchases > 0
  ORDER BY total_spent DESC
  LIMIT 10
""").show(truncate=False)

print("\n2. User segmentation:")
spark.sql(f"""
  SELECT 
    CASE 
      WHEN purchases > 0 THEN 'Buyer'
      WHEN total_events > 50 THEN 'Active'
      ELSE 'Casual'
    END as segment,
    COUNT(*) as user_count
  FROM delta.`{silver_path}`
  GROUP BY 1
  ORDER BY user_count DESC
""").show(truncate=False)

🔍 SQL Queries on Silver Layer (Direct Path):

1. Top buyers:
+---------+------------+---------+------------------+---------------+
|user_id  |total_events|purchases|total_spent       |avg_order_value|
+---------+------------+---------+------------------+---------------+
|512845454|3217        |3        |2664289.25        |888096.42      |
|513558661|1951        |1        |1956747.7499999993|1956747.75     |
|537873067|4627        |17       |1733347.0900000024|101961.59      |
|513900355|2288        |79       |1663103.7900000005|21051.95       |
|568805468|2847        |31       |1567113.0599999994|50552.03       |
|512386086|2511        |603      |1544661.5700000005|2561.63        |
|547556934|1109        |1        |1412989.9200000002|1412989.92     |
|568793129|4771        |1        |1390417.8700000006|1390417.87     |
|545925192|1552        |123      |1362327.9299999992|11075.84       |
|513068111|3183        |2        |1354343.4799999997|677171.74      |
+---------+------------+-----

In [0]:
final_count = spark.read.format("delta").load(silver_path).count()
print(f"✅ PRODUCTION READY: {final_count:,} clean user profiles")

✅ PRODUCTION READY: 5,316,650 clean user profiles
